In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
import re

In [2]:
df = pd.read_csv("steak.csv")
df.head()

,id,recipeName,rating,totalTimeInSeconds,course,cuisine,ingredients
0,Steak-Bites-1353195,Steak Bites,4,44100.0,NaN,NaN,"[rib eye steaks, kosher salt, ground black pep..."
1,Steak-Bites-1283837,Steak Bites,4,900.0,NaN,NaN,"[sirloin steak, kosher salt, ground black pepp..."
2,Herb-Crusted-Strip-Steak-1373651,Herb Crusted Strip Steak,4,2100.0,[Main Dishes],NaN,"[new york strip steaks, olive oil, salt, steak]"
3,Classic-Pan-Seared-Rib-Eye-Steak-1363563,Classic Pan-Seared Rib-Eye Steak,4,1800.0,[Main Dishes],NaN,"[rib eye steaks, peanut oil, coarse kosher sal..."
4,Slow-Cooked-Steak-with-Creamy-Mushroom-Sauce-1...,Slow Cooked Steak with Creamy Mushroom Sauce,4,18300.0,[Main Dishes],NaN,"[beef steak, beef stock, cooking cream, mushro..."


In [3]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 339 entries, 0 to 338
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  339 non-null    str    
 1   recipeName          339 non-null    str    
 2   rating              339 non-null    int64  
 3   totalTimeInSeconds  333 non-null    float64
 4   course              312 non-null    str    
 5   cuisine             74 non-null     str    
 6   ingredients         339 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 77.1 KB


id                      0
recipeName              0
rating                  0
totalTimeInSeconds      6
course                 27
cuisine               265
ingredients             0
dtype: int64

In [6]:
text_column = [col for col in df.columns if "text" in col.lower()]
print(text_column)

[]


In [7]:
df.head()

,id,recipeName,rating,totalTimeInSeconds,course,cuisine,ingredients
0,Steak-Bites-1353195,Steak Bites,4,44100.0,NaN,NaN,"[rib eye steaks, kosher salt, ground black pep..."
1,Steak-Bites-1283837,Steak Bites,4,900.0,NaN,NaN,"[sirloin steak, kosher salt, ground black pepp..."
2,Herb-Crusted-Strip-Steak-1373651,Herb Crusted Strip Steak,4,2100.0,[Main Dishes],NaN,"[new york strip steaks, olive oil, salt, steak]"
3,Classic-Pan-Seared-Rib-Eye-Steak-1363563,Classic Pan-Seared Rib-Eye Steak,4,1800.0,[Main Dishes],NaN,"[rib eye steaks, peanut oil, coarse kosher sal..."
4,Slow-Cooked-Steak-with-Creamy-Mushroom-Sauce-1...,Slow Cooked Steak with Creamy Mushroom Sauce,4,18300.0,[Main Dishes],NaN,"[beef steak, beef stock, cooking cream, mushro..."


In [9]:
print(df.head())
print(df.columns)

                                                  id  \
0                                Steak-Bites-1353195   
1                                Steak-Bites-1283837   
2                   Herb-Crusted-Strip-Steak-1373651   
3           Classic-Pan-Seared-Rib-Eye-Steak-1363563   
4  Slow-Cooked-Steak-with-Creamy-Mushroom-Sauce-1...   

                                     recipeName  rating  totalTimeInSeconds  \
0                                   Steak Bites       4             44100.0   
1                                   Steak Bites       4               900.0   
2                      Herb Crusted Strip Steak       4              2100.0   
3              Classic Pan-Seared Rib-Eye Steak       4              1800.0   
4  Slow Cooked Steak with Creamy Mushroom Sauce       4             18300.0   

          course cuisine                                        ingredients  
0            NaN     NaN  [rib eye steaks, kosher salt, ground black pep...  
1            NaN     NaN  [sirlo

In [11]:
print("text column:", "text" in df.columns)
print("clean_text column:", "clean_text" in df.columns)
print("X exists:", "X" in globals())

text column: False
clean_text column: False
X exists: False


In [12]:
df.head()

,id,recipeName,rating,totalTimeInSeconds,course,cuisine,ingredients
0,Steak-Bites-1353195,Steak Bites,4,44100.0,NaN,NaN,"[rib eye steaks, kosher salt, ground black pep..."
1,Steak-Bites-1283837,Steak Bites,4,900.0,NaN,NaN,"[sirloin steak, kosher salt, ground black pepp..."
2,Herb-Crusted-Strip-Steak-1373651,Herb Crusted Strip Steak,4,2100.0,[Main Dishes],NaN,"[new york strip steaks, olive oil, salt, steak]"
3,Classic-Pan-Seared-Rib-Eye-Steak-1363563,Classic Pan-Seared Rib-Eye Steak,4,1800.0,[Main Dishes],NaN,"[rib eye steaks, peanut oil, coarse kosher sal..."
4,Slow-Cooked-Steak-with-Creamy-Mushroom-Sauce-1...,Slow Cooked Steak with Creamy Mushroom Sauce,4,18300.0,[Main Dishes],NaN,"[beef steak, beef stock, cooking cream, mushro..."


In [14]:
print(df.columns)
print(df.head())

Index(['id', 'recipeName', 'rating', 'totalTimeInSeconds', 'course', 'cuisine',
       'ingredients'],
      dtype='str')
                                                  id  \
0                                Steak-Bites-1353195   
1                                Steak-Bites-1283837   
2                   Herb-Crusted-Strip-Steak-1373651   
3           Classic-Pan-Seared-Rib-Eye-Steak-1363563   
4  Slow-Cooked-Steak-with-Creamy-Mushroom-Sauce-1...   

                                     recipeName  rating  totalTimeInSeconds  \
0                                   Steak Bites       4             44100.0   
1                                   Steak Bites       4               900.0   
2                      Herb Crusted Strip Steak       4              2100.0   
3              Classic Pan-Seared Rib-Eye Steak       4              1800.0   
4  Slow Cooked Steak with Creamy Mushroom Sauce       4             18300.0   

          course cuisine                                        in

In [15]:
df.to_csv("steak_clustered.csv", index=False)